# WISER Module 6 Colab Template: Machine Learning for Addiction Research

        **Learning objective:** Fit, evaluate, and audit a simple prediction model while treating label definition, subgroup performance, and deployment accountability as core parts of the workflow.

> **Before you begin:** In Google Colab, choose **File → Save a copy in Drive**. Work only in your copy.
>
> This notebook connects to your prior WISER work. If you completed earlier modules, you may import outputs from those notebooks where prompted.
>
> **Privacy reminder:** Do not upload protected health information, identifiable clinical notes, or non-approved institutional data. Use the WISER synthetic datasets unless your instructor has explicitly approved another dataset.

In [ ]:
MODULE_ID = "Module 6"
NOTEBOOK_TITLE = "WISER Module 6: Machine Learning for Addiction Research"

## Setup

In [ ]:
from pathlib import Path
import json
import textwrap
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

PALETTE = {
    "teal": "#20808D",
    "rust": "#A84B2F",
    "dark_teal": "#1B474D",
    "light_cyan": "#BCE2E7",
    "mauve": "#944454",
    "gold": "#FFC553",
}

STIGMA_TERMS = {
    "addict": "person with a substance use disorder",
    "addicts": "people with substance use disorders",
    "abuser": "person with a substance use disorder",
    "drug abuse": "substance use or substance use disorder",
    "clean": "negative toxicology / not currently using",
    "dirty": "positive toxicology",
    "junkie": "person with a substance use disorder",
}

def stigma_audit(text):
    '''Return potentially stigmatizing terms found in a string.'''
    text_lower = str(text).lower()
    return {term: suggestion for term, suggestion in STIGMA_TERMS.items() if term in text_lower}

def safe_label(text):
    '''Lightweight learner-facing label checker. Human review is still required.'''
    hits = stigma_audit(text)
    if hits:
        print("Review this wording:")
        for term, suggestion in hits.items():
            print(f" - Replace '{term}' with '{suggestion}' when clinically appropriate.")
    return text

def quick_profile(df, name="dataset"):
    '''Basic data profile for WISER learner notebooks.'''
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    profile = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_n": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(1),
        "unique_n": df.nunique(dropna=True),
    })
    display(profile)
    display(df.head())

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Section 1: Data Loading and Exploration

In [ ]:
DATA_URL = ""  # TODO: paste final WISER hopebridge_synthetic_ml.csv URL, if available

if DATA_URL:
    ml = pd.read_csv(DATA_URL)
else:
    n = 600
    rng = np.random.default_rng(RANDOM_SEED)
    ml = pd.DataFrame({
        "age": rng.integers(18, 70, n),
        "race_ethnicity": rng.choice(["Black", "Hispanic/Latino", "White", "Other"], n, p=[.28, .18, .45, .09]),
        "gender": rng.choice(["Woman", "Man", "Nonbinary/Other"], n, p=[.46, .49, .05]),
        "insurance": rng.choice(["Medicaid", "Commercial", "Uninsured"], n, p=[.55, .30, .15]),
        "housing_insecure": rng.choice([0, 1], n, p=[.70, .30]),
        "prior_treatment_episodes": rng.poisson(1.5, n),
        "phq9": rng.normal(10, 5, n).clip(0, 27).round(0),
        "gad7": rng.normal(8, 4, n).clip(0, 21).round(0),
        "dsm5_severity": rng.choice(["mild", "moderate", "severe"], n, p=[.25, .40, .35]),
        "svi_quartile": rng.choice([1, 2, 3, 4], n),
    })
    logit = (
        -2.0
        + 0.45 * ml["housing_insecure"]
        + 0.18 * ml["prior_treatment_episodes"]
        + 0.04 * ml["phq9"]
        + 0.22 * (ml["insurance"] == "Uninsured")
        + 0.16 * (ml["svi_quartile"] >= 3)
    )
    p = 1 / (1 + np.exp(-logit))
    ml["return_to_use_6mo"] = rng.binomial(1, p)

quick_profile(ml, "hopebridge_synthetic_ml")
ml["return_to_use_6mo"].value_counts(normalize=True).rename("prevalence")

## Section 2: Label Construction Check

In a real project, the outcome label must be co-defined with clinical stakeholders and people with lived experience. This template uses `return_to_use_6mo` as a synthetic teaching label.

In [ ]:
label = "return_to_use_6mo"
features = ["age", "race_ethnicity", "gender", "insurance", "housing_insecure", "prior_treatment_episodes", "phq9", "gad7", "dsm5_severity", "svi_quartile"]
display(ml[[label] + features].head())

## Section 3: Train/Test Split and Baseline Model

In [ ]:
X = ml[features]
y = ml[label]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_SEED, stratify=y
)

numeric_features = ["age", "prior_treatment_episodes", "phq9", "gad7", "svi_quartile"]
categorical_features = ["race_ethnicity", "gender", "insurance", "housing_insecure", "dsm5_severity"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

model.fit(X_train, y_train)
pred_proba = model.predict_proba(X_test)[:, 1]
pred = (pred_proba >= 0.5).astype(int)

print("Test AUC:", round(roc_auc_score(y_test, pred_proba), 3))
print(classification_report(y_test, pred))

## Section 4: Fairness and Subgroup Audit

In [ ]:
audit = X_test.copy()
audit["y_true"] = y_test.values
audit["y_score"] = pred_proba
audit["y_pred"] = pred

def subgroup_metrics(df, group_col):
    rows = []
    for group, g in df.groupby(group_col):
        if g["y_true"].nunique() < 2:
            auc = np.nan
        else:
            auc = roc_auc_score(g["y_true"], g["y_score"])
        tn, fp, fn, tp = confusion_matrix(g["y_true"], g["y_pred"], labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
        specificity = tn / (tn + fp) if (tn + fp) else np.nan
        rows.append({
            "group_variable": group_col,
            "group": group,
            "n": len(g),
            "auc": round(auc, 3) if pd.notna(auc) else np.nan,
            "sensitivity": round(sensitivity, 3),
            "specificity": round(specificity, 3),
        })
    return pd.DataFrame(rows)

fairness = pd.concat([
    subgroup_metrics(audit, "race_ethnicity"),
    subgroup_metrics(audit, "gender"),
    subgroup_metrics(audit, "insurance"),
], ignore_index=True)

display(fairness)
fairness.to_csv(OUTPUT_DIR / "module6_fairness_audit.csv", index=False)

## Section 5: Foundational Pathway Reflection

In 150–250 words, explain whether you would deploy this model, deploy with conditions, or not deploy. Use at least two metrics from the subgroup audit.

In [ ]:
deployment_memo = """TODO: Write your deployment recommendation here."""
print(textwrap.fill(deployment_memo, 90))

## Section 6: Applied Pathway Extension

In [ ]:
# Applied task ideas:
# 1. Try a DecisionTreeClassifier or RandomForestClassifier.
# 2. Change the probability threshold and compare subgroup sensitivity.
# 3. Remove SVI and compare whether performance/fairness changes.
print("Applied extension: add a comparison model below and re-run the fairness audit.")

## Section 7: FAIR Export and Submission

In [ ]:
metadata = {
    "module": MODULE_ID,
    "notebook": NOTEBOOK_TITLE,
    "created_by": "WISER learner",
    "uses_synthetic_or_public_data": True,
    "contains_phi": False,
    "fair_notes": {
        "findable": "Outputs saved with clear names in outputs/.",
        "accessible": "Notebook and exported files can be shared through the course portal.",
        "interoperable": "CSV/JSON/Markdown formats used where possible.",
        "reusable": "README and metadata describe inputs, assumptions, and outputs.",
    },
}

(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))
(OUTPUT_DIR / "README.md").write_text(f'''# {NOTEBOOK_TITLE} Outputs

This folder contains learner-generated outputs for {MODULE_ID}.

## Files

- `metadata.json`: FAIR-oriented metadata
- Additional module-specific artifacts produced by notebook cells

## Privacy

This template is designed for synthetic or public data. Do not include PHI or identifiable institutional data.
''')

print("FAIR export complete.")
print("Generated:", sorted(str(p) for p in OUTPUT_DIR.iterdir()))